## earthkit-data: streams

In [1]:
import earthkit.data as ekd

## Loading URL as a stream

We read GRIB data from a URL as a stream. With this we can avoid disk usage and can work entirely in memory.

In [2]:
url = "https://sites.ecmwf.int/repository/earthkit/samples/test6.grib"
d = ekd.from_source("url", url, stream=True)
d

types,fieldlist


The usage of this data is rather limited, be we can convert it into a stream fieldlist and iterate through it field by field. This is a special fieldlist, apart from the iteration no other methods are available, e.g.len() does not work.

In [3]:
fl = d.to_fieldlist()
try:
    len(fl)
except Exception as e:
    print(e)

StreamFieldList does not support __len__


When we iterate through the stream fieldlist Field objects are created then get deleted when going out of scope. As a result, only one GRIB message is kept in memory at a time.

In [4]:
for f in fl:
    print(f)

Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(u, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(u, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)


When we finished the iteration we consumed all the data from the stream. Another attemp to iterate would yield nothing.

In [5]:
for f in fl:
    print(f)

### Using parts

We can use parts (byte ranges) for URLs and it also works for streams.

In [6]:
# Only read the bytes for the ranges of 0-240 and 480-960 from the URL
# This contains GRIB message 0, 2 and 3.
# The part specification has the format of (start, length)
d = ekd.from_source(
        "url",
        url,
        parts=[(0, 240), (480, 480)], stream=True)

for f in d.to_fieldlist():
    print(f)

Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)


### Reading the whole stream into memory

In [7]:
# fl is a fieldlist entirely in memory
d = ekd.from_source("url", url, stream=True)
fl = d.to_fieldlist(read_all=True)
fl.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
2,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
3,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
4,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
5,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


Please use this option carefully!